In [0]:
%python
import pandas as pd
import requests
from io import StringIO

url = "https://raw.githubusercontent.com/01Vishwa/IBM-HR-Attrition-Dashboard/main/HR-Employee-Attrition.csv"

print("Baixando dataset via HTTP...")

response = requests.get(url, timeout=30)
response.raise_for_status()

pdf_hr = pd.read_csv(StringIO(response.text))

# Converte todas as colunas object para string puro (evita erro do Arrow)
for col in pdf_hr.select_dtypes(include=["object"]).columns:
    pdf_hr[col] = pdf_hr[col].astype(str)

df_hr = spark.createDataFrame(pdf_hr)
df_hr.createOrReplaceTempView("hr_data")

print(f"Sucesso! {df_hr.count()} registros carregados na view 'hr_data'.")
df_hr.printSchema()

Baixando dataset via HTTP...
Sucesso! 1470 registros carregados na view 'hr_data'.
root
 |-- Age: long (nullable = true)
 |-- Attrition: string (nullable = true)
 |-- BusinessTravel: string (nullable = true)
 |-- DailyRate: long (nullable = true)
 |-- Department: string (nullable = true)
 |-- DistanceFromHome: long (nullable = true)
 |-- Education: long (nullable = true)
 |-- EducationField: string (nullable = true)
 |-- EmployeeCount: long (nullable = true)
 |-- EmployeeNumber: long (nullable = true)
 |-- EnvironmentSatisfaction: long (nullable = true)
 |-- Gender: string (nullable = true)
 |-- HourlyRate: long (nullable = true)
 |-- JobInvolvement: long (nullable = true)
 |-- JobLevel: long (nullable = true)
 |-- JobRole: string (nullable = true)
 |-- JobSatisfaction: long (nullable = true)
 |-- MaritalStatus: string (nullable = true)
 |-- MonthlyIncome: long (nullable = true)
 |-- MonthlyRate: long (nullable = true)
 |-- NumCompaniesWorked: long (nullable = true)
 |-- Over18: string

In [0]:
%sql
-- 2.1 Volume total e contagem de atrito
SELECT 
  COUNT(*) AS total_funcionarios,
  SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS total_saidas,
  ROUND(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS taxa_atrito_pct
FROM hr_data

total_funcionarios,total_saidas,taxa_atrito_pct
1470,237,16.12


In [0]:
%sql
-- 2.2 Visão geral das colunas-chave (distribuição rápida)
SELECT 
  Department,
  JobRole,
  ROUND(AVG(MonthlyIncome), 2) AS media_salario,
  ROUND(AVG(Age), 1) AS media_idade,
  ROUND(AVG(DistanceFromHome), 1) AS media_distancia_km,
  ROUND(AVG(YearsAtCompany), 1) AS media_anos_empresa,
  COUNT(*) AS qtd
FROM hr_data
GROUP BY Department, JobRole
ORDER BY Department, qtd DESC

Department,JobRole,media_salario,media_idade,media_distancia_km,media_anos_empresa,qtd
Human Resources,Human Resources,4235.75,35.5,8.2,5.3,52
Human Resources,Manager,18088.64,48.7,11.2,16.3,11
Research & Development,Research Scientist,3239.97,34.2,9.0,5.1,292
Research & Development,Laboratory Technician,3237.17,34.1,9.4,5.0,259
Research & Development,Manufacturing Director,7295.14,38.3,9.5,7.6,145
Research & Development,Healthcare Representative,7528.76,39.8,9.8,8.4,131
Research & Development,Research Director,16033.55,44.0,8.4,10.9,80
Research & Development,Manager,17130.33,46.0,7.2,13.5,54
Sales,Sales Executive,6924.28,36.9,9.7,7.5,326
Sales,Sales Representative,2626.0,30.4,8.7,2.9,83


In [0]:
%sql
-- 2.3 Checagem de valores nulos nas colunas críticas
SELECT
  SUM(CASE WHEN Attrition IS NULL THEN 1 ELSE 0 END) AS nulls_attrition,
  SUM(CASE WHEN MonthlyIncome IS NULL THEN 1 ELSE 0 END) AS nulls_salario,
  SUM(CASE WHEN Department IS NULL THEN 1 ELSE 0 END) AS nulls_departamento,
  SUM(CASE WHEN Age IS NULL THEN 1 ELSE 0 END) AS nulls_idade,
  SUM(CASE WHEN DistanceFromHome IS NULL THEN 1 ELSE 0 END) AS nulls_distancia
FROM hr_data

nulls_attrition,nulls_salario,nulls_departamento,nulls_idade,nulls_distancia
0,0,0,0,0


In [0]:
%sql
-- 3. Turnover por Departamento
SELECT 
  Department AS departamento,
  COUNT(*) AS total_funcionarios,
  SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS saidas,
  ROUND(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS taxa_atrito_pct
FROM hr_data
GROUP BY Department
ORDER BY taxa_atrito_pct DESC

departamento,total_funcionarios,saidas,taxa_atrito_pct
Sales,446,92,20.63
Human Resources,63,12,19.05
Research & Development,961,133,13.84


In [0]:
%sql
-- 4. Turnover por Faixa Salarial
SELECT 
  CASE 
    WHEN MonthlyIncome < 3000 THEN '1. Até 3k'
    WHEN MonthlyIncome BETWEEN 3000 AND 5999 THEN '2. 3k–6k'
    WHEN MonthlyIncome BETWEEN 6000 AND 9999 THEN '3. 6k–10k'
    WHEN MonthlyIncome BETWEEN 10000 AND 14999 THEN '4. 10k–15k'
    ELSE '5. Acima de 15k'
  END AS faixa_salarial,
  COUNT(*) AS total,
  SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS saidas,
  ROUND(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS taxa_atrito_pct
FROM hr_data
GROUP BY faixa_salarial
ORDER BY faixa_salarial

faixa_salarial,total,saidas,taxa_atrito_pct
1. Até 3k,395,113,28.61
2. 3k–6k,519,66,12.72
3. 6k–10k,275,33,12.00
4. 10k–15k,148,20,13.51
5. Acima de 15k,133,5,3.76


In [0]:
%sql
-- 5.1 Turnover por Faixa de Distância de Casa
SELECT 
  CASE 
    WHEN DistanceFromHome <= 5 THEN '1. Perto (0–5)'
    WHEN DistanceFromHome BETWEEN 6 AND 15 THEN '2. Média (6–15)'
    ELSE '3. Longe (16+)'
  END AS faixa_distancia,
  COUNT(*) AS total,
  SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS saidas,
  ROUND(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS taxa_atrito_pct
FROM hr_data
GROUP BY faixa_distancia
ORDER BY faixa_distancia

faixa_distancia,total,saidas,taxa_atrito_pct
1. Perto (0–5),632,87,13.77
2. Média (6–15),509,82,16.11
3. Longe (16+),329,68,20.67


In [0]:
%sql
SELECT 
  CASE JobSatisfaction
    WHEN 1 THEN '1. Baixa'
    WHEN 2 THEN '2. Média'
    WHEN 3 THEN '3. Alta'
    WHEN 4 THEN '4. Muito Alta'
  END AS nivel_satisfacao,
  COUNT(*) AS total,
  SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS saidas,
  ROUND(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS taxa_atrito_pct
FROM hr_data
GROUP BY JobSatisfaction
ORDER BY JobSatisfaction

nivel_satisfacao,total,saidas,taxa_atrito_pct
1. Baixa,289,66,22.84
2. Média,280,46,16.43
3. Alta,442,73,16.52
4. Muito Alta,459,52,11.33


In [0]:
%sql
SELECT 
  CASE 
    WHEN YearsAtCompany <= 1 THEN '1. Até 1 ano'
    WHEN YearsAtCompany BETWEEN 2 AND 5 THEN '2. 2–5 anos'
    WHEN YearsAtCompany BETWEEN 6 AND 10 THEN '3. 6–10 anos'
    ELSE '4. Mais de 10 anos'
  END AS faixa_tempo_casa,
  COUNT(*) AS total,
  SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS saidas,
  ROUND(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS taxa_atrito_pct
FROM hr_data
GROUP BY faixa_tempo_casa
ORDER BY faixa_tempo_casa

faixa_tempo_casa,total,saidas,taxa_atrito_pct
1. Até 1 ano,215,75,34.88
2. 2–5 anos,561,87,15.51
3. 6–10 anos,448,55,12.28
4. Mais de 10 anos,246,20,8.13


In [0]:
%sql
-- 6. Base para scatter plot: cada ponto = 1 funcionário
SELECT 
  Age AS idade,
  MonthlyIncome AS renda_mensal,
  Attrition AS atrito,
  Department AS departamento,
  JobRole AS cargo,
  YearsAtCompany AS anos_empresa,
  JobSatisfaction AS satisfacao
FROM hr_data
ORDER BY Attrition DESC

idade,renda_mensal,atrito,departamento,cargo,anos_empresa,satisfacao
41,5993,Yes,Sales,Sales Executive,6,4
37,2090,Yes,Research & Development,Laboratory Technician,0,3
28,2028,Yes,Research & Development,Laboratory Technician,4,3
36,3407,Yes,Sales,Sales Representative,5,1
34,2960,Yes,Research & Development,Research Scientist,4,1
32,3919,Yes,Research & Development,Research Scientist,10,1
39,2086,Yes,Sales,Sales Representative,1,4
24,2293,Yes,Research & Development,Research Scientist,2,4
50,2683,Yes,Sales,Sales Representative,3,3
26,2293,Yes,Research & Development,Laboratory Technician,1,3


In [0]:
%sql
-- 7. Ranking dos fatores de risco combinados
SELECT 
  Department AS departamento,
  JobRole AS cargo,
  CASE 
    WHEN MonthlyIncome < 3000 THEN 'Baixo'
    WHEN MonthlyIncome BETWEEN 3000 AND 6999 THEN 'Médio'
    ELSE 'Alto'
  END AS nivel_salarial,
  CASE 
    WHEN YearsAtCompany <= 1 THEN 'Novato'
    WHEN YearsAtCompany BETWEEN 2 AND 5 THEN 'Intermediário'
    ELSE 'Veterano'
  END AS senioridade,
  COUNT(*) AS total,
  SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS saidas,
  ROUND(SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS taxa_atrito_pct
FROM hr_data
GROUP BY departamento, cargo, nivel_salarial, senioridade
HAVING COUNT(*) >= 5
ORDER BY taxa_atrito_pct DESC
LIMIT 15

departamento,cargo,nivel_salarial,senioridade,total,saidas,taxa_atrito_pct
Human Resources,Human Resources,Baixo,Novato,5,4,80.00
Sales,Sales Representative,Baixo,Novato,30,17,56.67
Research & Development,Laboratory Technician,Baixo,Novato,45,22,48.89
Sales,Sales Representative,Médio,Intermediário,11,5,45.45
Research & Development,Laboratory Technician,Médio,Novato,16,6,37.50
Sales,Sales Executive,Alto,Novato,8,3,37.50
Human Resources,Human Resources,Baixo,Intermediário,15,5,33.33
Sales,Sales Representative,Baixo,Intermediário,31,10,32.26
Research & Development,Research Scientist,Baixo,Novato,42,13,30.95
Research & Development,Research Scientist,Baixo,Veterano,38,9,23.68
